In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import time
import json
from tqdm import tqdm

# Инициализация MediaPipe
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

# Индексы для Face Mesh
LEFT_EYE = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
RIGHT_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
LEFT_IRIS = [474, 475, 476, 477]
RIGHT_IRIS = [469, 470, 471, 472]

### --- МЕТОД 1: Текущий (Baseline из Charisma Master) ---

In [ ]:
def analyze_gaze_baseline(face_landmarks):
    # Копия логики из ml_engine.py
    face = face_landmarks.landmark
    nose_x = face[1].x
    left_ear = face[234].x
    right_ear = face[454].x

    face_width = abs(right_ear - left_ear)
    face_center = (left_ear + right_ear) / 2

    if face_width > 0:
        deviation = abs(nose_x - face_center) / face_width
        return 1.0 if deviation < 0.25 else 0.0
    return 0.0

### --- МЕТОД 2: Face Mesh Head Pose (Углы поворота головы) ---

In [ ]:
def get_head_pose_v2(face_landmarks, img_w, img_h):
    # 1 - кончик носа, 152 - подбородок, 33 - левый угол левого глаза,
    # 263 - правый угол правого глаза, 61 - левый угол рта, 291 - правый угол рта
    face_2d = []
    indices = [1, 152, 33, 263, 61, 291]

    # Примерные 3D координаты модели лица
    model_3d = np.array([
        [0.0, 0.0, 0.0],          # Нос
        [0.0, -330.0, -65.0],    # Подбородок
        [-225.0, 170.0, -135.0], # Левый глаз
        [225.0, 170.0, -135.0],  # Правый глаз
        [-150.0, -150.0, -125.0],# Левый рот
        [150.0, -150.0, -125.0]  # Правый рот
    ], dtype=np.float64)

    for idx in indices:
        lm = face_landmarks.landmark[idx]
        face_2d.append([lm.x * img_w, lm.y * img_h])

    face_2d = np.array(face_2d, dtype=np.float64)
    focal_length = img_w
    cam_matrix = np.array([[focal_length, 0, img_w / 2],
                           [0, focal_length, img_h / 2],
                           [0, 0, 1]])

    success, rot_vec, trans_vec = cv2.solvePnP(model_3d, face_2d, cam_matrix, np.zeros((4, 1)))
    rmat, _ = cv2.Rodrigues(rot_vec)

    # Прямой расчет углов
    pitch = np.arcsin(-rmat[2, 0])
    yaw = np.arctan2(rmat[2, 1], rmat[2, 2])

    return np.degrees(pitch), np.degrees(yaw)

### --- МЕТОД 3: Тяжелый (Face Mesh + Iris Tracking + Pose Orientation) ---

In [ ]:
def get_gaze_ratio(face_landmarks, eye_indices, iris_indices):
    lm = face_landmarks.landmark
    # Берем X-координаты краев глаза
    # eye_indices[0] - внутренний угол, eye_indices[8] - внешний угол
    x_left = lm[eye_indices[0]].x
    x_right = lm[eye_indices[8]].x
    
    # Средний X зрачка
    x_iris = np.mean([lm[i].x for i in iris_indices])
    
    # Вычисляем, где зрачок относительно углов глаза (0.0 - 1.0)
    width = abs(x_right - x_left)
    if width == 0: return 0.5
    
    # Находим относительную позицию
    ratio = (x_iris - min(x_left, x_right)) / width
    return ratio

def analyze_gaze_advanced(face_landmarks, pose_landmarks, pitch, raw_yaw, img_w, img_h):
    # 1. Нормализация Yaw (180 -> 0)
    yaw = raw_yaw - 180 if raw_yaw > 0 else raw_yaw + 180
    
    # 2. Зрачки (avg_gaze)
    ratio_l = get_gaze_ratio(face_landmarks, LEFT_EYE, LEFT_IRIS)
    ratio_r = get_gaze_ratio(face_landmarks, RIGHT_EYE, RIGHT_IRIS)
    avg_gaze = (ratio_l + ratio_r) / 2

    # 3. ВОЗВРАЩАЕМ ДОПУСКИ ДЛЯ ЦЕНТРА (чтобы оживить Профи и Контакт)
    # Расширяем Pitch до 60 (как в твоих логах), Yaw до 20
    is_head_centered = (abs(yaw) < 20) and (-20 < pitch < 65)
    
    # Возвращаем комфортный диапазон для зрачков (0.35 - 0.65)
    # Это позволит Профи и Full Contact снова получить свои 0.9+
    is_gaze_centered = (0.35 < avg_gaze < 0.65)
    
    # 4. УМНАЯ КОМПЕНСАЦИЯ
    # Мы засчитываем контакт при повороте головы ТОЛЬКО если глаза 
    # смотрят в противоположную сторону от поворота головы.
    compensation = False
    
    # Если голова повернута ВПРАВО (yaw < -20)
    if yaw < -20:
        # Глаза должны смотреть ВЛЕВО (в сторону камеры), т.е. avg_gaze должен уменьшиться
        if avg_gaze < 0.38: 
            compensation = True
            
    # Если голова повернута ВЛЕВО (yaw > 20)
    elif yaw > 20:
        # Глаза должны смотреть ВПРАВО, т.е. avg_gaze должен увеличиться
        if avg_gaze > 0.62:
            compensation = True

    # Итог: Либо всё по центру, либо есть корректная компенсация
    if is_head_centered and is_gaze_centered:
        return True
    return compensation



In [ ]:
def process_video_full_suite(video_path):
    cap = cv2.VideoCapture(video_path)
    results = {"baseline_hits": 0, "mesh_hits": 0, "advanced_hits": 0, "frames_with_face": 0, "total_processed_frames": 0}

    with mp_face_mesh.FaceMesh(refine_landmarks=True) as face_mesh, mp_pose.Pose() as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            results["total_processed_frames"] += 1
            if results["total_processed_frames"] % 5 != 0: continue

            h, w, _ = frame.shape
            face_results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if face_results.multi_face_landmarks:
                results["frames_with_face"] += 1
                face_lms = face_results.multi_face_landmarks[0]
                results["baseline_hits"] += analyze_gaze_baseline(face_lms)

                pitch, raw_yaw = get_head_pose_v2(face_lms, w, h)

                yaw_c = raw_yaw - 180 if raw_yaw > 0 else raw_yaw + 180

                if abs(yaw_c) < 15 and (-15 < pitch < 40):
                    results["mesh_hits"] += 1

                if results["frames_with_face"] % 20 == 0:
                     print(f"Debug [{video_path[-15:]}]: Pitch {pitch:.1f}, Yaw_с {yaw_c:.1f}, Raw_yaw {raw_yaw:.1f}")

                if analyze_gaze_advanced(face_lms, None, pitch, raw_yaw, w, h): 
                    results["advanced_hits"] += 1

    cap.release()
    f = max(1, results["frames_with_face"])
    for k in ["baseline", "mesh", "advanced"]: results[f"score_{k}"] = results[f"{k}_hits"] / f
    return results

In [ ]:
dataset = [
    {"path": "media/custom_contact.mp4", "label": "ideal"},
    {"path": "media/custom_faceright_contact.mp4", "label": "ideal_face-right"},
    {"path": "media/custom_rotate_cantact.mp4", "label": "ideal_rotate"},
    {"path": "media/expert_bad_light.mp4", "label": "expert_bad-light"},
    {"path": "media/expert_glasses.mp4", "label": "expert_glasses"},
    {"path": "media/custom_facedown_nocontact.mp4", "label": "bad_face-down"},
    {"path": "media/no_eye_contact.mp4", "label": "bad"},
    {"path": "media/custom_faceright_nocontact.mp4", "label": "bad_face-right"},
]

### запуск


In [ ]:
all_stats = []
for video in dataset:
    try:
        print(f"Processing {video['path']}...")
        res = process_video_full_suite(video['path'])
        res.update({'video': video['path'], 'label': video['label']})
        all_stats.append(res)
    except Exception as e: print(f"Error: {e}")

display(pd.DataFrame(all_stats))